In [1]:
from dotenv import load_dotenv
import os
import requests
from datetime import date, timedelta
from pprint import pprint

load_dotenv('apis.env')
API_KEY = os.getenv('API')

#### API 1: NEOWS


In [2]:
feed_start_date = date.today() + timedelta(days=1)
feed_end_date = feed_start_date + timedelta(days=6)

params = {
    'start_date': feed_start_date.isoformat(),
    'end_date': feed_end_date.isoformat(),
    'api_key': API_KEY,
}
response = requests.get(
    'https://api.nasa.gov/neo/rest/v1/feed', params=params, timeout=30
)
response.raise_for_status()
data = response.json()

In [3]:
if 'near_earth_objects' not in data:
    raise RuntimeError(f"NeoWs returned an unexpected response: {data}")

print(
    f"NeoWs returned {data['element_count']} asteroids from "
    f"{feed_start_date.isoformat()} through {feed_end_date.isoformat()}."
)

NeoWs returned 43 asteroids from 2026-07-20 through 2026-07-26.


In [4]:
asteroids_by_id = {}

for approach_date, asteroids in data['near_earth_objects'].items():
    for neo in asteroids:
        asteroid_id = neo['id']
        asteroid = asteroids_by_id.setdefault(
            asteroid_id,
            {
                'name': neo['name'],
                'id': asteroid_id,
                'estimated_diameter_meters': {
                    'min': neo['estimated_diameter']['meters']['estimated_diameter_min'],
                    'max': neo['estimated_diameter']['meters']['estimated_diameter_max'],
                },
                'is_hazardous': neo['is_potentially_hazardous_asteroid'],
                'is_sentry_object': neo['is_sentry_object'],
                'close_approach_data': [],
            },
        )

        for approach in neo.get('close_approach_data', []):
            if approach.get('orbiting_body') == 'Earth':
                asteroid['close_approach_data'].append(
                    {
                        'date': approach.get('close_approach_date', approach_date),
                        'date_full': approach.get('close_approach_date_full'),
                        'miss_distance_km': float(approach['miss_distance']['kilometers']),
                        'relative_velocity_km_s': float(
                            approach['relative_velocity']['kilometers_per_second']
                        ),
                    }
                )

todays_asteroids_data = list(asteroids_by_id.values())
print(f"Prepared {len(todays_asteroids_data)} unique asteroids for enrichment.")

Prepared 43 unique asteroids for enrichment.


### API 2: JPL Small-Body Database (SBDB)


In [5]:
SBDB_API = 'https://ssd-api.jpl.nasa.gov/sbdb.api'

def get_orbit_value(elements, target_name):
    for element in elements:
        if element.get('name') == target_name and element.get('value') is not None:
            return float(element['value'])
    return None

def optional_float(value):
    return float(value) if value not in (None, '', 'n/a') else None

def optional_int(value):
    return int(value) if value not in (None, '', 'n/a') else None

for asteroid_data in todays_asteroids_data:
    response = requests.get(SBDB_API, params={'spk': asteroid_data['id']}, timeout=30)
    response.raise_for_status()
    sbdb_data = response.json()

    if 'error' in sbdb_data:
        asteroid_data['sbdb_error'] = sbdb_data['error']
        continue

    object_data = sbdb_data['object']
    orbit_data = sbdb_data['orbit']
    orbit_class = object_data.get('orbit_class') or {}

    asteroid_data.update(
        {
            'spk_id': optional_int(object_data.get('spkid')),
            'designation': object_data.get('des'),
            'pha': object_data.get('pha'),
            'orbit_class_name': orbit_class.get('name'),
            'semi_major_axis_au': get_orbit_value(orbit_data['elements'], 'a'),
            'eccentricity': get_orbit_value(orbit_data['elements'], 'e'),
            'inclination_degrees': get_orbit_value(orbit_data['elements'], 'i'),
            'longitude_of_ascending_node_degrees': get_orbit_value(
                orbit_data['elements'], 'om'
            ),
            'argument_of_perihelion_degrees': get_orbit_value(
                orbit_data['elements'], 'w'
            ),
            'mean_anomaly_degrees': get_orbit_value(orbit_data['elements'], 'ma'),
            'orbital_period_days': get_orbit_value(orbit_data['elements'], 'per'),
            'moid_au': optional_float(orbit_data.get('moid')),
        }
    )
print(f"Enriched {len(todays_asteroids_data)} asteroids with SBDB data.")

Enriched 43 asteroids with SBDB data.


### API 3: JPL Horizons VECTORS


In [ ]:
import re

HORIZONS_API = 'https://ssd.jpl.nasa.gov/api/horizons.api'
SUN_CENTER = '500@10'

def extract_horizons_vectors(result_text):
    lines = result_text.splitlines()
    try:
        start = next(i for i, line in enumerate(lines) if line.strip() == '$$SOE') + 1
        end = next(i for i, line in enumerate(lines) if line.strip() == '$$EOE')
    except StopIteration as error:
        raise ValueError('Horizons response did not contain vector data.') from error

    number_pattern = r'[-+]?(?:\d+\.?\d*|\.\d+)[ED][+-]\d+'
    vectors = []
    data_lines = lines[start:end]

    for index, line in enumerate(data_lines):
        if 'A.D.' not in line:
            continue

        jd = float(line.split('=')[0].strip())
        datetime_string = line.split('A.D.')[1].split('TDB')[0].strip()
        position_values = re.findall(number_pattern, data_lines[index + 1])
        velocity_values = re.findall(number_pattern, data_lines[index + 2])

        if len(position_values) != 3 or len(velocity_values) != 3:
            raise ValueError('Horizons returned an incomplete state vector.')

        x, y, z = (float(value.replace('D', 'E')) for value in position_values)
        vx, vy, vz = (float(value.replace('D', 'E')) for value in velocity_values)
        vectors.append(
            {
                'jd': jd,
                'datetime': datetime_string,
                'x_km': x,
                'y_km': y,
                'z_km': z,
                'vx_kms': vx,
                'vy_kms': vy,
                'vz_kms': vz,
            }
        )

    return vectors

horizons_start = feed_start_date.strftime('%Y-%m-%d 00:00')
horizons_stop = (feed_end_date + timedelta(days=1)).strftime('%Y-%m-%d 00:00')

def fetch_horizons_vectors(command):
    params = {
        'format': 'json',
        'EPHEM_TYPE': 'VECTORS',
        'COMMAND': command,
        'CENTER': SUN_CENTER,
        'START_TIME': f"'{horizons_start}'",
        'STOP_TIME': f"'{horizons_stop}'",
        'STEP_SIZE': '1h',
        'REF_PLANE': 'ECLIPTIC',
        'REF_SYSTEM': 'ICRF',
        'OUT_UNITS': 'KM-S',
        'VEC_TABLE': '2',
        'TIME_TYPE': 'TDB',
        'TIME_DIGITS': 'SECONDS',
    }
    response = requests.get(HORIZONS_API, params=params, timeout=60)
    response.raise_for_status()
    return extract_horizons_vectors(response.json()['result'])

for asteroid_data in todays_asteroids_data:
    if asteroid_data.get('spk_id') is None:
        continue

    asteroid_data['vectors'] = fetch_horizons_vectors(
        f"'DES={asteroid_data['spk_id']};'"
    )

planet_commands = {
    'Mercury': '199',
    'Venus': '299',
    'Earth': '399',
    'Mars': '499',
    'Jupiter': '599',
    'Saturn': '699',
    'Uranus': '799',
    'Neptune': '899',
}

planets_data = {
    planet_name: fetch_horizons_vectors(f"'{planet_command}'")
    for planet_name, planet_command in planet_commands.items()
}

print(f"Added Sun-centered state vectors for {len(todays_asteroids_data)} asteroids.")
print(f"Added Sun-centered state vectors for {len(planets_data)} planets.")

### API 4: JPL Sentry API


In [7]:
SENTRY_API = 'https://ssd-api.jpl.nasa.gov/sentry.api'

for asteroid_data in todays_asteroids_data:
    if not asteroid_data['is_sentry_object']:
        asteroid_data['sentry'] = None
        continue

    response = requests.get(
        SENTRY_API, params={'spk': asteroid_data['spk_id']}, timeout=30
    )
    response.raise_for_status()
    sentry_data = response.json()

    if 'error' in sentry_data:
        asteroid_data['sentry'] = {
            'status': 'removed' if 'removed' in sentry_data else 'not_found',
            'error': sentry_data['error'],
            'removed_date': sentry_data.get('removed'),
        }
        continue

    summary = sentry_data.get('summary', {})
    potential_impacts = [
        {
            'date': impact.get('date'),
            'impact_probability': optional_float(impact.get('ip')),
            'palermo_scale': optional_float(impact.get('ps')),
            'torino_scale': optional_int(impact.get('ts')),
            'impact_energy_megatons': optional_float(impact.get('energy')),
        }
        for impact in sentry_data.get('data', [])
    ]

    asteroid_data['sentry'] = {
        'status': 'available',
        'impact_probability': optional_float(summary.get('ip')),
        'palermo_scale_cumulative': optional_float(summary.get('ps_cum')),
        'palermo_scale_max': optional_float(summary.get('ps_max')),
        'torino_scale_max': optional_int(summary.get('ts_max')),
        'impact_energy_megatons': optional_float(summary.get('energy')),
        'potential_impact_dates': [impact['date'] for impact in potential_impacts],
        'potential_impacts': potential_impacts,
    }

sentry_count = sum(asteroid['is_sentry_object'] for asteroid in todays_asteroids_data)
print(f"Added Sentry data for {sentry_count} monitored asteroids.")

Added Sentry data for 1 monitored asteroids.


## View asteroid data

1. Run all of the API cells above.
2. Use the next cell to see each asteroid's number.
3. Run `show_asteroid(number)` to print one asteroid's complete data.

For example, `show_asteroid(0)` shows the first asteroid in the list.


In [8]:
def list_asteroids():
    """Print a simple numbered list so you can choose an asteroid index."""
    if not globals().get('todays_asteroids_data'):
        print('No asteroid data yet. Run the API cells above first.')
        return

    for index, asteroid in enumerate(todays_asteroids_data):
        approaches = asteroid.get('close_approach_data', [])
        approach_date = approaches[0]['date'] if approaches else 'unknown date'
        print(f"[{index}] {asteroid['name']} — close approach: {approach_date}")

list_asteroids()


[0] 524474 (2002 KJ3) — close approach: 2026-07-20
[1] (2002 AJ69) — close approach: 2026-07-20
[2] (2009 HC) — close approach: 2026-07-20
[3] (2017 VZ13) — close approach: 2026-07-20
[4] (2018 PR10) — close approach: 2026-07-20
[5] (2019 NG2) — close approach: 2026-07-20
[6] (2019 VK3) — close approach: 2026-07-20
[7] (2019 WG) — close approach: 2026-07-20
[8] (2004 YC) — close approach: 2026-07-21
[9] (2007 YH) — close approach: 2026-07-21
[10] (2016 WB8) — close approach: 2026-07-21
[11] (2020 BO12) — close approach: 2026-07-21
[12] (2020 BX12) — close approach: 2026-07-21
[13] (2020 DA1) — close approach: 2026-07-21
[14] 523728 (2014 ON344) — close approach: 2026-07-22
[15] (2007 SW2) — close approach: 2026-07-22
[16] (2015 EG7) — close approach: 2026-07-22
[17] (2015 TO237) — close approach: 2026-07-22
[18] (2015 VC105) — close approach: 2026-07-22
[19] 253062 (2002 TC70) — close approach: 2026-07-23
[20] 448628 (2010 VF1) — close approach: 2026-07-23
[21] (2014 GJ1) — close appro

In [9]:
def show_asteroid(index):
    """Print all stored data for one asteroid. Example: show_asteroid(0)."""
    if not globals().get('todays_asteroids_data'):
        print('No asteroid data yet. Run the API cells above first.')
        return

    if not isinstance(index, int):
        print('Use a whole number, for example: show_asteroid(0)')
        return

    if index < 0 or index >= len(todays_asteroids_data):
        print(f"Choose an index from 0 to {len(todays_asteroids_data) - 1}.")
        return

    pprint(todays_asteroids_data[index], sort_dicts=False)

# Change 0 to any number shown by list_asteroids().
show_asteroid(0)


{'name': '524474 (2002 KJ3)',
 'id': '2524474',
 'estimated_diameter_meters': {'min': 363.5423218434, 'max': 812.9053443399},
 'is_hazardous': False,
 'is_sentry_object': False,
 'close_approach_data': [{'date': '2026-07-20',
                          'date_full': '2026-Jul-20 19:11',
                          'miss_distance_km': 28253000.994950823,
                          'relative_velocity_km_s': 4.6120243861}],
 'spk_id': 20524474,
 'designation': '524474',
 'pha': False,
 'orbit_class_name': 'Amor',
 'semi_major_axis_au': 2.27,
 'eccentricity': 0.489,
 'inclination_degrees': 6.4,
 'longitude_of_ascending_node_degrees': 47.4,
 'argument_of_perihelion_degrees': 253.0,
 'mean_anomaly_degrees': 346.0,
 'orbital_period_days': 1250.0,
 'moid_au': 0.183,
 'vectors': [{'jd': 2461241.5,
              'datetime': '2026-Jul-20 00:00:00.0000',
              'x_km': 2592360.628559455,
              'y_km': -21750754.24852152,
              'z_km': -17849511.05386405,
              'vx_kms': 4